# DeepSecure-Forensics: MeGA-Based Genetic Weight Merging for Robust Audio Deepfake Detection
## MeGA-IA v2 — Genetic Algorithm Weight Merging Notebook

**Pipeline overview:**
1. Load pre-trained parent model weights from disk
2. Initialise a population of 15 merged candidates (layer-wise)
3. Evolve for 10 generations with fitness = Balanced Accuracy
4. Save a checkpoint after every generation
5. Calibrate classification threshold on val, evaluate on test
6. Save the final child model

**Key changes vs MeGA-IA v1:**
- ✅ CKA / entropy penalties removed — fitness is pure Balanced Accuracy
- ✅ Layer-wise population initialisation (3 group αs per individual)
- ✅ Annealed sigma: `σ_max=0.15 → σ_min=0.01` over generations
- ✅ Steeper depth-adaptive decay: `alpha_decay = 2.0` (was 0.5)
- ✅ Calibrated F1-optimal threshold at test time
- ✅ Full checkpoint saving after every generation

---
## Cell 1 — Install & Import Dependencies

In [ ]:
# ── Install tqdm if not already present ──────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tqdm', '-q'])

# ── Standard library ─────────────────────────────────────────────────────────
import os
import json
import time
import random
import datetime

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import tensorflow as tf
from tqdm.notebook import tqdm, trange
from sklearn.metrics import (
    balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam

# ── Project model import ──────────────────────────────────────────────────────
# Adjust this import to match your actual model definition
import model.resnet as resnets

print('✅ All imports successful')
print(f'   TensorFlow : {tf.__version__}')
print(f'   NumPy      : {np.__version__}')
print(f'   GPU devices: {tf.config.list_physical_devices("GPU")}')

---
## Cell 2 — Configuration

In [ ]:
# =============================================================================
# CONFIGURATION — Edit these paths and hyperparameters before running
# =============================================================================

# ── Paths ─────────────────────────────────────────────────────────────────────
PARENT_1_WEIGHTS_PATH = 'weights/parent_model_1.weights.h5'   # ← set your path
PARENT_2_WEIGHTS_PATH = 'weights/parent_model_2.weights.h5'   # ← set your path
CHECKPOINT_DIR        = 'checkpoints/mega_ia_v2/'             # generation checkpoints
FINAL_MODEL_PATH      = 'models/mega_ia_v2_child_model.keras' # final saved model
LOG_PATH              = 'logs/mega_ia_v2_run.json'            # GA run log (JSON)

# ── GA Hyperparameters ────────────────────────────────────────────────────────
POPULATION_SIZE  = 15     # number of individuals per generation
NUM_GENERATIONS  = 10     # number of GA generations
NUM_PARENTS      = 4      # tournament draws per offspring
MUTATION_RATE    = 0.02   # per-layer mutation probability

# ── Mutation annealing ────────────────────────────────────────────────────────
SIGMA_MAX    = 0.15   # exploration noise at generation 0
SIGMA_MIN    = 0.01   # exploitation noise at final generation
ALPHA_DECAY  = 2.0    # depth-adaptive decay (higher = more head protection)

# ── Fitness threshold (fixed during GA; calibrated post-convergence) ──────────
THRESHOLD_DURING_GA = 0.5

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 46

# ── Create output directories ─────────────────────────────────────────────────
for d in [CHECKPOINT_DIR, os.path.dirname(FINAL_MODEL_PATH), os.path.dirname(LOG_PATH)]:
    if d:
        os.makedirs(d, exist_ok=True)

print('✅ Configuration set')
print(f'   Population size : {POPULATION_SIZE}')
print(f'   Generations     : {NUM_GENERATIONS}')
print(f'   Mutation rate   : {MUTATION_RATE}')
print(f'   σ schedule      : {SIGMA_MAX} → {SIGMA_MIN}')
print(f'   Alpha decay     : {ALPHA_DECAY}')
print(f'   Checkpoint dir  : {CHECKPOINT_DIR}')
print(f'   Final model     : {FINAL_MODEL_PATH}')

---
## Cell 3 — Reproducibility & Utilities

In [ ]:
def seed_everything(seed):
    """Fix all randomness sources for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    tf.random.set_seed(seed)
    print(f'🔒 Seed set to {seed}')


def timestamp():
    return datetime.datetime.now().strftime('%H:%M:%S')


def log_section(title):
    """Pretty-print a section header."""
    bar = '═' * 60
    print(f'\n{bar}')
    print(f'  {title}')
    print(f'{bar}')


seed_everything(SEED)

---
## Cell 4 — Model Builder

In [ ]:
def build_model():
    """
    Constructs and compiles the base model.
    Adjust architecture and input_shape to match your parents.
    This is the single source of truth for model creation throughout the notebook.
    """
    model = resnets.ResNet56(input_shape=(32, 32, 3), num_classes=10)
    model.compile(
        optimizer=Adam(learning_rate=0.01),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# Build a dummy model to confirm architecture loads
_probe = build_model()
num_layers = len(_probe.get_weights())
del _probe

print(f'✅ Model architecture verified')
print(f'   Total weight tensors (layers): {num_layers}')
print(f'   Layer groups (⌊L/3⌋)         : early=[0,{num_layers//3}), '
      f'mid=[{num_layers//3},{2*num_layers//3}), '
      f'late=[{2*num_layers//3},{num_layers})')

---
## Cell 5 — Load Parent Model Weights

In [ ]:
log_section('STEP 1 — Loading Parent Model Weights')

# ── Parent 1 ──────────────────────────────────────────────────────────────────
print(f'[{timestamp()}] Loading parent 1 from: {PARENT_1_WEIGHTS_PATH}')
parent_model_1 = build_model()
parent_model_1.load_weights(PARENT_1_WEIGHTS_PATH)
weights1 = parent_model_1.get_weights()
print(f'[{timestamp()}] ✅ Parent 1 loaded — {len(weights1)} weight tensors')

# ── Parent 2 ──────────────────────────────────────────────────────────────────
print(f'[{timestamp()}] Loading parent 2 from: {PARENT_2_WEIGHTS_PATH}')
parent_model_2 = build_model()
parent_model_2.load_weights(PARENT_2_WEIGHTS_PATH)
weights2 = parent_model_2.get_weights()
print(f'[{timestamp()}] ✅ Parent 2 loaded — {len(weights2)} weight tensors')

# ── Weight compatibility check ────────────────────────────────────────────────
assert len(weights1) == len(weights2), (
    f'❌ Parent weight mismatch: parent1 has {len(weights1)} tensors, '
    f'parent2 has {len(weights2)}'
)
for i, (w1, w2) in enumerate(zip(weights1, weights2)):
    assert w1.shape == w2.shape, (
        f'❌ Shape mismatch at tensor {i}: {w1.shape} vs {w2.shape}'
    )

print(f'\n✅ Weight compatibility verified across all {len(weights1)} tensors')
total_params = sum(w.size for w in weights1)
print(f'   Total parameters per model: {total_params:,}')

---
## Cell 6 — Data Loading & Validation Split
> **Note:** Replace the CIFAR-10 block below with your actual audio deepfake dataset loading.

In [ ]:
log_section('STEP 2 — Data Loading')

# =============================================================================
# ⚠️  REPLACE THIS BLOCK with your DF40 / ASVspoof dataset loading.
#     x_val must be your validation features (shape: [N, ...]).
#     y_val must be flat integer labels shape: [N,].
#     Binary:     0=real, 1=fake
#     Multi-class: 0..K-1
# =============================================================================
from tensorflow.keras.datasets import cifar10

print(f'[{timestamp()}] Loading dataset...')
(x_train_full, y_train_full), (x_test, y_test) = cifar10.load_data()
x_train_full = x_train_full.astype('float32') / 255.0
x_test       = x_test.astype('float32') / 255.0

# 90/10 train/val split
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.1, random_state=SEED
)

# Flat integer labels for sklearn metrics
y_val_flat  = y_val.ravel().astype(int)
y_test_flat = y_test.ravel().astype(int)

print(f'[{timestamp()}] ✅ Data loaded')
print(f'   Train  : {x_train.shape}  labels: {y_train.shape}')
print(f'   Val    : {x_val.shape}    labels: {y_val_flat.shape}')
print(f'   Test   : {x_test.shape}   labels: {y_test_flat.shape}')
print(f'   Classes: {np.unique(y_val_flat).tolist()}')

IS_BINARY = len(np.unique(y_val_flat)) == 2
print(f'   Task   : {"Binary (deepfake)" if IS_BINARY else "Multi-class"}')

---
## Cell 7 — Evaluate Parent Models on Validation Set

In [ ]:
log_section('STEP 3 — Parent Model Baseline Evaluation')

for idx, (model, name) in enumerate([
    (parent_model_1, 'Parent 1'),
    (parent_model_2, 'Parent 2')
], 1):
    print(f'[{timestamp()}] Evaluating {name} on val set...')
    _, val_acc = model.evaluate(x_val, y_val, verbose=0)

    preds = model.predict(x_val, verbose=0)
    y_pred = np.argmax(preds, axis=1) if preds.shape[-1] > 2 else (preds[:, 1] >= 0.5).astype(int)
    bal_acc = balanced_accuracy_score(y_val_flat, y_pred)

    print(f'   {name} — Val Accuracy: {val_acc:.4f} | Val Balanced Acc: {bal_acc:.4f}')

print(f'\n[{timestamp()}] ✅ Parent baselines recorded. GA will aim to beat these.')

---
## Cell 8 — MeGA-IA v2 Core Functions

In [ ]:
# =============================================================================
# MeGA-IA v2 Core — Fitness, Population, Selection, Crossover, Mutation
# =============================================================================

# ── Fitness: Balanced Accuracy ────────────────────────────────────────────────
def evaluate_fitness(model, weights, x_val, y_val_flat, threshold=0.5):
    """
    Fitness = Balanced Accuracy = 0.5·(TPR + TNR).

    Replaces v1's F_IA = Acc − λ₁·I − λ₂·H.
    CKA and entropy penalties are removed entirely.

    Binary tasks  → applies `threshold` to positive-class probability.
    Multi-class   → argmax predictions, threshold ignored.
    """
    model.set_weights(weights)
    preds_prob = model.predict(x_val, verbose=0)

    if preds_prob.shape[-1] == 2:
        y_pred = (preds_prob[:, 1] >= threshold).astype(int)
    else:
        y_pred = np.argmax(preds_prob, axis=1)

    return float(balanced_accuracy_score(y_val_flat, y_pred))


# ── Calibrated threshold search ───────────────────────────────────────────────
def find_optimal_threshold(model, x_val, y_val_flat):
    """
    Searches τ* ∈ [0.01, 0.99] that maximises F1 on the validation set.
    Applied once after GA convergence.
    Returns 0.5 sentinel for multi-class (argmax is used instead).
    """
    preds_prob = model.predict(x_val, verbose=0)
    if preds_prob.shape[-1] > 2:
        return 0.5

    pos_prob  = preds_prob[:, 1]
    best_tau  = 0.5
    best_f1   = -1.0

    for tau in np.arange(0.01, 1.00, 0.01):
        y_pred = (pos_prob >= tau).astype(int)
        f1 = f1_score(y_val_flat, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1  = f1
            best_tau = float(tau)

    return best_tau


# ── Layer-wise population initialisation ──────────────────────────────────────
def create_population(size, weights1, weights2):
    """
    Initialises population with three independent αs per individual
    (one per layer group: early / mid / late).

    v1 used a single global α for all layers, killing initial diversity.
    """
    L      = len(weights1)
    third  = L // 3
    pop    = []

    for _ in range(size):
        a_early = random.random()
        a_mid   = random.random()
        a_late  = random.random()

        individual = []
        for l, (w1, w2) in enumerate(zip(weights1, weights2)):
            alpha = a_early if l < third else (a_mid if l < 2 * third else a_late)
            individual.append((1.0 - alpha) * w1 + alpha * w2)

        pop.append(individual)

    return pop


# ── Tournament selection ──────────────────────────────────────────────────────
def tournament_selection(population, fitnesses, k):
    """Draws k parents via 3-way tournament selection."""
    selected = []
    for _ in range(k):
        contenders = random.sample(list(zip(population, fitnesses)), 3)
        contenders.sort(key=lambda x: x[1], reverse=True)
        selected.append(contenders[0][0])
    return selected


# ── Layer-wise adaptive crossover ─────────────────────────────────────────────
def compute_layer_variance_scores(weights):
    """s_l = Var(θ_l) / (‖θ_l‖² + ε) — knowledge richness per layer."""
    return [float(np.var(w)) / (float(np.sum(w**2)) + 1e-8) for w in weights]


def adaptive_crossover(p1_weights, p2_weights):
    """
    Layer-wise crossover: β_l = s_l(p1)/(s_l(p1)+s_l(p2)) + δ_l
    δ_l ~ Uniform(−0.1, 0.1) for diversity.
    """
    s1 = compute_layer_variance_scores(p1_weights)
    s2 = compute_layer_variance_scores(p2_weights)

    child = []
    for w1, w2, v1, v2 in zip(p1_weights, p2_weights, s1, s2):
        beta  = v1 / (v1 + v2 + 1e-8)
        delta = np.random.uniform(-0.1, 0.1)
        beta  = float(np.clip(beta + delta, 0.0, 1.0))
        child.append(beta * w1 + (1.0 - beta) * w2)

    return child


# ── Depth-adaptive mutation ───────────────────────────────────────────────────
def depth_adaptive_mutation(individual, mutation_rate, sigma_base, alpha_decay):
    """
    σ_l = sigma_base · exp(−alpha_decay · l / L)

    alpha_decay=2.0 → deepest layer gets only ~9% of base noise,
    near-freezing the classification head.
    """
    L = len(individual)
    mutated = []
    for l, w in enumerate(individual):
        sigma_l = sigma_base * np.exp(-alpha_decay * l / L)
        if random.random() < mutation_rate:
            mutated.append(w + np.random.normal(0.0, sigma_l, w.shape))
        else:
            mutated.append(w.copy())
    return mutated


# ── Sigma annealing ───────────────────────────────────────────────────────────
def anneal_sigma(generation, num_generations, sigma_max, sigma_min):
    """
    Geometric annealing: σ_g = σ_max · (σ_min/σ_max)^(g/G)
    Exploration early, exploitation late.
    """
    ratio = sigma_min / sigma_max
    return float(sigma_max * (ratio ** (generation / max(num_generations - 1, 1))))


print('✅ All MeGA-IA v2 functions defined')

---
## Cell 9 — Checkpoint Utilities

In [ ]:
def save_checkpoint(generation, best_weights, best_fitness, history, model):
    """
    Saves a full checkpoint after each generation:
      - Best individual weights (.weights.h5)
      - GA run history so far (.json)

    Args:
        generation   (int): current generation (1-based)
        best_weights (list[np.ndarray]): best weight tensors so far
        best_fitness (float): best balanced accuracy so far
        history      (list[dict]): log of all generations so far
        model        (tf.keras.Model): vessel to load weights into before saving
    """
    # ── Weight checkpoint ─────────────────────────────────────────────────────
    ckpt_weights_path = os.path.join(
        CHECKPOINT_DIR, f'gen_{generation:02d}_best.weights.h5'
    )
    model.set_weights(best_weights)
    model.save_weights(ckpt_weights_path)

    # ── History JSON ──────────────────────────────────────────────────────────
    ckpt_log_path = os.path.join(
        CHECKPOINT_DIR, f'gen_{generation:02d}_history.json'
    )
    with open(ckpt_log_path, 'w') as f:
        json.dump({
            'generation'   : generation,
            'best_fitness' : best_fitness,
            'timestamp'    : timestamp(),
            'history'      : history
        }, f, indent=2)

    print(f'   💾 Checkpoint saved → {ckpt_weights_path}')
    print(f'   📋 History saved    → {ckpt_log_path}')

    return ckpt_weights_path


print('✅ Checkpoint functions ready')
print(f'   Checkpoints will be saved to: {CHECKPOINT_DIR}')

---
## Cell 10 — Population Initialisation

In [ ]:
log_section('STEP 4 — Population Initialisation')

print(f'[{timestamp()}] Creating initial population of {POPULATION_SIZE} individuals...')
print(f'   Method: layer-wise 3-group α sampling')
print(f'   Layer groups: early / mid / late (each gets independent α ~ U(0,1))')

t0 = time.time()
population = create_population(POPULATION_SIZE, weights1, weights2)
elapsed = time.time() - t0

print(f'[{timestamp()}] ✅ Population created in {elapsed:.2f}s')
print(f'   Individuals : {len(population)}')
print(f'   Tensors each: {len(population[0])}')

# Diversity sanity check — average pairwise L2 distance between individuals 0 and 1
diff = np.mean([np.linalg.norm(a - b) for a, b in zip(population[0], population[1])])
print(f'   Avg L2 dist (ind[0] vs ind[1]): {diff:.6f}  (>0 = diverse ✅)')

# Initialise tracking variables
best_individual = None
best_fitness    = -np.inf
ga_history      = []          # per-generation log
checkpoint_paths = []         # paths to saved checkpoints

# Shared evaluation vessel (single model instance reused to save GPU memory)
model_fusion = build_model()
print(f'\n[{timestamp()}] Evaluation vessel (model_fusion) built and ready')

---
## Cell 11 — MeGA-IA v2 Main GA Loop

In [ ]:
log_section('STEP 5 — MeGA-IA v2 Genetic Algorithm')

print(f'[{timestamp()}] GA config summary:')
print(f'   Population  : {POPULATION_SIZE} individuals')
print(f'   Generations : {NUM_GENERATIONS}')
print(f'   Fitness     : Balanced Accuracy  (threshold={THRESHOLD_DURING_GA})')
print(f'   Crossover   : Layer-wise adaptive (variance score β_l)')
print(f'   Mutation    : Depth-adaptive (α_decay={ALPHA_DECAY})')
print(f'   Σ annealing : {SIGMA_MAX} → {SIGMA_MIN}')
print(f'   Elitism     : ON (global best always carried forward)')
print(f'   Checkpoints : every generation → {CHECKPOINT_DIR}')
print()

# ── Generation loop ───────────────────────────────────────────────────────────
gen_bar = trange(1, NUM_GENERATIONS + 1, desc='Generations', unit='gen')

for generation in gen_bar:
    gen_start = time.time()

    # ── Sigma for this generation ─────────────────────────────────────────────
    sigma_g = anneal_sigma(
        generation - 1, NUM_GENERATIONS, SIGMA_MAX, SIGMA_MIN
    )

    # ── Fitness evaluation ────────────────────────────────────────────────────
    fitnesses = []
    ind_bar = tqdm(
        population,
        desc=f'  Gen {generation:02d} — evaluating',
        leave=False,
        unit='ind'
    )
    for individual in ind_bar:
        fit = evaluate_fitness(
            model_fusion, individual,
            x_val, y_val_flat,
            threshold=THRESHOLD_DURING_GA
        )
        fitnesses.append(fit)
        ind_bar.set_postfix({'last_bal_acc': f'{fit:.4f}'})

    # ── Update global best ────────────────────────────────────────────────────
    gen_best_idx     = int(np.argmax(fitnesses))
    gen_best_fitness = fitnesses[gen_best_idx]
    gen_mean_fitness = float(np.mean(fitnesses))
    gen_worst        = float(np.min(fitnesses))
    improved         = False

    if gen_best_fitness > best_fitness:
        best_fitness    = gen_best_fitness
        best_individual = [w.copy() for w in population[gen_best_idx]]
        improved        = True

    # ── Log this generation ───────────────────────────────────────────────────
    gen_elapsed = time.time() - gen_start
    gen_record  = {
        'generation'        : generation,
        'best_bal_acc_gen'  : round(gen_best_fitness, 6),
        'mean_bal_acc_gen'  : round(gen_mean_fitness, 6),
        'worst_bal_acc_gen' : round(gen_worst, 6),
        'global_best'       : round(best_fitness, 6),
        'sigma_base'        : round(sigma_g, 6),
        'improved'          : improved,
        'elapsed_sec'       : round(gen_elapsed, 2),
        'timestamp'         : timestamp()
    }
    ga_history.append(gen_record)

    # ── Save checkpoint ───────────────────────────────────────────────────────
    ckpt_path = save_checkpoint(
        generation, best_individual, best_fitness, ga_history, model_fusion
    )
    checkpoint_paths.append(ckpt_path)

    # ── Console log ───────────────────────────────────────────────────────────
    improved_tag = ' ⬆ NEW BEST' if improved else ''
    gen_bar.set_postfix({
        'global_best': f'{best_fitness:.4f}',
        'gen_best'   : f'{gen_best_fitness:.4f}',
        'σ'          : f'{sigma_g:.4f}'
    })
    print(
        f'\n[{timestamp()}] Gen {generation:02d}/{NUM_GENERATIONS} | '
        f'Best={gen_best_fitness:.4f} | '
        f'Mean={gen_mean_fitness:.4f} | '
        f'Worst={gen_worst:.4f} | '
        f'GlobalBest={best_fitness:.4f} | '
        f'σ={sigma_g:.4f} | '
        f'{gen_elapsed:.1f}s{improved_tag}'
    )

    # ── Build next generation ─────────────────────────────────────────────────
    next_population = [best_individual]   # elitism: global best survives

    offspring_bar = tqdm(
        range(POPULATION_SIZE - 1),
        desc=f'  Gen {generation:02d} — breeding',
        leave=False,
        unit='offspring'
    )
    for _ in offspring_bar:
        parents = tournament_selection(population, fitnesses, NUM_PARENTS)
        child   = adaptive_crossover(parents[0], parents[1])
        child   = depth_adaptive_mutation(
            child, MUTATION_RATE, sigma_g, ALPHA_DECAY
        )
        next_population.append(child)

    population = next_population

print(f'\n[{timestamp()}] ✅ GA complete — {NUM_GENERATIONS} generations evolved')
print(f'   Global best balanced accuracy : {best_fitness:.6f}')
print(f'   Total checkpoints saved       : {len(checkpoint_paths)}')

---
## Cell 12 — Post-GA Threshold Calibration

In [ ]:
log_section('STEP 6 — Calibrated Threshold Search')

model_fusion.set_weights(best_individual)

print(f'[{timestamp()}] Searching for F1-optimal threshold τ* on val set...')
print(f'   Search grid: τ ∈ [0.01, 0.99]  step=0.01')

tau_star = find_optimal_threshold(model_fusion, x_val, y_val_flat)

print(f'[{timestamp()}] ✅ Calibration complete')
if IS_BINARY:
    print(f'   Optimal threshold τ* = {tau_star:.2f}  (applied at test time)')
else:
    print(f'   Multi-class task — argmax used (τ* sentinel = {tau_star})')

---
## Cell 13 — Final Evaluation

In [ ]:
log_section('STEP 7 — Final Test Set Evaluation')

model_fusion.set_weights(best_individual)

# ── Predictions ───────────────────────────────────────────────────────────────
print(f'[{timestamp()}] Running inference on test set...')
test_preds_prob = model_fusion.predict(x_test, verbose=1)

if test_preds_prob.shape[-1] == 2:
    y_pred_test = (test_preds_prob[:, 1] >= tau_star).astype(int)
else:
    y_pred_test = np.argmax(test_preds_prob, axis=1)

# ── Metrics ───────────────────────────────────────────────────────────────────
test_bal_acc   = balanced_accuracy_score(y_test_flat, y_pred_test)
_, test_acc    = model_fusion.evaluate(x_test, y_test, verbose=0)

# Parent baselines (re-evaluated for comparison)
_, p1_acc = parent_model_1.evaluate(x_test, y_test, verbose=0)
_, p2_acc = parent_model_2.evaluate(x_test, y_test, verbose=0)
p1_preds  = np.argmax(parent_model_1.predict(x_test, verbose=0), axis=1)
p2_preds  = np.argmax(parent_model_2.predict(x_test, verbose=0), axis=1)
p1_bal    = balanced_accuracy_score(y_test_flat, p1_preds)
p2_bal    = balanced_accuracy_score(y_test_flat, p2_preds)

# ── Results table ─────────────────────────────────────────────────────────────
print(f'\n{"═"*60}')
print(f'  MeGA-IA v2 — Final Results')
print(f'{"═"*60}')
print(f'  {"Model":<30} {"Accuracy":>10} {"Balanced Acc":>14}')
print(f'  {"─"*56}')
print(f'  {"Parent 1":<30} {p1_acc:>10.4f} {p1_bal:>14.4f}')
print(f'  {"Parent 2":<30} {p2_acc:>10.4f} {p2_bal:>14.4f}')
print(f'  {"─"*56}')
print(f'  {"MeGA-IA v2 Child (τ*=" + str(round(tau_star,2)) + ")":<30} {test_acc:>10.4f} {test_bal_acc:>14.4f}')
print(f'{"═"*60}')

# Delta vs best parent
best_parent_bal = max(p1_bal, p2_bal)
delta = test_bal_acc - best_parent_bal
sign  = '+' if delta >= 0 else ''
print(f'  Δ Balanced Acc vs best parent: {sign}{delta:.4f}')
print(f'  Global best val Balanced Acc : {best_fitness:.6f}')
print(f'{"═"*60}')

# ── Classification report ─────────────────────────────────────────────────────
print(f'\n[{timestamp()}] Classification Report (test set):')
print(classification_report(y_test_flat, y_pred_test, zero_division=0))

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test_flat, y_pred_test)
print(f'Confusion Matrix:')
print(cm)

---
## Cell 14 — Save Final Child Model

In [ ]:
log_section('STEP 8 — Saving Final Child Model')

model_fusion.set_weights(best_individual)

# ── Save full model (.keras) ──────────────────────────────────────────────────
print(f'[{timestamp()}] Saving full child model to: {FINAL_MODEL_PATH}')
model_fusion.save(FINAL_MODEL_PATH)
print(f'[{timestamp()}] ✅ Full model saved')

# ── Save weights only ─────────────────────────────────────────────────────────
weights_only_path = FINAL_MODEL_PATH.replace('.keras', '.weights.h5')
model_fusion.save_weights(weights_only_path)
print(f'[{timestamp()}] ✅ Weights saved → {weights_only_path}')

# ── Save full GA run log ──────────────────────────────────────────────────────
final_log = {
    'run_info': {
        'population_size' : POPULATION_SIZE,
        'num_generations' : NUM_GENERATIONS,
        'mutation_rate'   : MUTATION_RATE,
        'alpha_decay'     : ALPHA_DECAY,
        'sigma_max'       : SIGMA_MAX,
        'sigma_min'       : SIGMA_MIN,
        'seed'            : SEED,
        'threshold_ga'    : THRESHOLD_DURING_GA,
        'tau_star'        : tau_star
    },
    'results': {
        'parent_1_test_acc'       : round(p1_acc, 6),
        'parent_2_test_acc'       : round(p2_acc, 6),
        'parent_1_test_bal_acc'   : round(p1_bal, 6),
        'parent_2_test_bal_acc'   : round(p2_bal, 6),
        'child_test_acc'          : round(test_acc, 6),
        'child_test_bal_acc'      : round(test_bal_acc, 6),
        'best_val_bal_acc_ga'     : round(best_fitness, 6),
        'delta_vs_best_parent'    : round(delta, 6)
    },
    'generation_history' : ga_history,
    'checkpoint_paths'   : checkpoint_paths
}

os.makedirs(os.path.dirname(LOG_PATH) if os.path.dirname(LOG_PATH) else '.', exist_ok=True)
with open(LOG_PATH, 'w') as f:
    json.dump(final_log, f, indent=2)

print(f'[{timestamp()}] ✅ Full GA log saved → {LOG_PATH}')

print(f'\n{"═"*60}')
print(f'  MeGA-IA v2 Run Complete')
print(f'{"═"*60}')
print(f'  Child model : {FINAL_MODEL_PATH}')
print(f'  Weights     : {weights_only_path}')
print(f'  GA log      : {LOG_PATH}')
print(f'  Checkpoints : {CHECKPOINT_DIR}  ({len(checkpoint_paths)} files)')
print(f'{"═"*60}')

---
## Cell 15 — GA Convergence Plot

In [ ]:
import matplotlib.pyplot as plt

gens        = [r['generation']        for r in ga_history]
best_curve  = [r['global_best']       for r in ga_history]
mean_curve  = [r['mean_bal_acc_gen']  for r in ga_history]
worst_curve = [r['worst_bal_acc_gen'] for r in ga_history]
sigmas      = [r['sigma_base']        for r in ga_history]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
fig.suptitle('MeGA-IA v2 — GA Convergence', fontsize=14, fontweight='bold')

# ── Fitness plot ──────────────────────────────────────────────────────────────
ax1.fill_between(gens, worst_curve, best_curve, alpha=0.15, color='steelblue', label='Worst–Best range')
ax1.plot(gens, best_curve,  'o-', color='steelblue',  linewidth=2, markersize=6, label='Global best')
ax1.plot(gens, mean_curve,  's--', color='darkorange', linewidth=1.5, markersize=5, label='Gen mean')
ax1.plot(gens, worst_curve, 'v:', color='gray',       linewidth=1, markersize=4, label='Gen worst')
ax1.set_ylabel('Balanced Accuracy', fontsize=11)
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title('Fitness (Balanced Accuracy) across Generations')

# Mark improved generations
for r in ga_history:
    if r['improved']:
        ax1.axvline(r['generation'], color='green', alpha=0.3, linestyle='--', linewidth=1)

# ── Sigma plot ────────────────────────────────────────────────────────────────
ax2.plot(gens, sigmas, 'D-', color='crimson', linewidth=2, markersize=6)
ax2.set_ylabel('σ_base (mutation noise)', fontsize=11)
ax2.set_xlabel('Generation', fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_title('Sigma Annealing Schedule (Exploration → Exploitation)')

plt.tight_layout()
plot_path = os.path.join(CHECKPOINT_DIR, 'convergence_plot.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Convergence plot saved → {plot_path}')